# Lean/Mathlib VPS cache for Colab

This notebook restores a prebuilt Lean/Mathlib environment from the Hong Kong VPS cache and runs a Lean verification on Colab local disk. It does not mount Google Drive.

Cache source: `https://lean-cache.yitwah.site/lean-mathlib-cache.tar.zst`

In [ ]:
%%bash
set -euxo pipefail
apt-get update -y
apt-get install -y curl zstd ca-certificates coreutils


In [ ]:
import os, pathlib

CACHE_BASE = "https://lean-cache.yitwah.site"
ARCHIVE_URL = f"{CACHE_BASE}/lean-mathlib-cache.tar.zst"
SHA_URL = f"{ARCHIVE_URL}.sha256"
ARCHIVE = pathlib.Path("/content/lean-mathlib-cache.tar.zst")
SHA_FILE = pathlib.Path("/content/lean-mathlib-cache.tar.zst.sha256")
PROJECT = pathlib.Path("/content/lean-cache-workspace/bruhat_mathlib")
ELAN_HOME = pathlib.Path("/content/.elan")

os.environ["ELAN_HOME"] = str(ELAN_HOME)
os.environ["PATH"] = f"{ELAN_HOME}/bin:" + os.environ["PATH"]
print("Archive:", ARCHIVE_URL)
print("Project:", PROJECT)


In [ ]:
%%bash
set -euxo pipefail
cd /content
curl -L --fail --retry 3 --retry-delay 5 -o lean-mathlib-cache.tar.zst https://lean-cache.yitwah.site/lean-mathlib-cache.tar.zst
curl -L --fail --retry 3 --retry-delay 5 -o lean-mathlib-cache.tar.zst.sha256 https://lean-cache.yitwah.site/lean-mathlib-cache.tar.zst.sha256
expected=$(awk '{print $1}' lean-mathlib-cache.tar.zst.sha256)
actual=$(sha256sum lean-mathlib-cache.tar.zst | awk '{print $1}')
test "$expected" = "$actual"
ls -lh lean-mathlib-cache.tar.zst


In [ ]:
%%bash
set -euxo pipefail
rm -rf /content/lean-cache-workspace /content/.elan
tar --zstd -xf /content/lean-mathlib-cache.tar.zst -C /content
test -x /content/.elan/bin/lean
test -d /content/lean-cache-workspace/bruhat_mathlib
/content/.elan/bin/lean --version


In [ ]:
import os, pathlib, subprocess, textwrap

os.environ["ELAN_HOME"] = "/content/.elan"
os.environ["PATH"] = "/content/.elan/bin:" + os.environ["PATH"]
project = pathlib.Path("/content/lean-cache-workspace/bruhat_mathlib")
smoke = project / "BruhatColabSmoke.lean"
smoke.write_text(textwrap.dedent("""\
import Mathlib

example (n : Nat) : n = n := rfl
"""))
subprocess.run(["lean", "--version"], check=True)
subprocess.run(["lake", "env", "lean", "BruhatColabSmoke.lean"], cwd=project, check=True)
print("LEAN_MATHLIB_OK")
